# Rough Heston calibration — Legendre

Single-run calibration to the rough-Heston synthetic surface, using the settings
from `Real data/Legendre/Calibration.ipynb`: two Brownian drivers, a Legendre
basis with **P=3, M=10**, and the same Heston initialization, optimizer, stopping
rules, scheduler, control variates, and computational chunks. The shortest
calibration maturity now uses **2 million paths** for both training and independent
evaluation to reduce Monte Carlo noise in deep out-of-the-money option prices.

The Heston starting model is fitted to the **calibration quotes only**. The
rough-Heston generating parameters are not used for initialization. Seven held-out
maturities measure how the calibrated model prices outside the fitted maturities;
they do not affect fitting, stopping, or model selection.

Use **Restart Kernel → Run All**. Runtime and accuracy are measured by this run;
the SPX results do not establish either for the rough-Heston surface.


## 1. Imports and rough-Heston data

Load `Rough Heston/rough_heston_data.pkl` and keep its calibration and test surfaces
separate. Paths work from the project root or this notebook's directory.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import pickle
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Rough Heston" / "Legendre"
NOTEBOOK_PATH = STUDY_DIR / "Calibration.ipynb"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.experiments.provenance import source_fingerprint
from wiener_chaos.pricing.options import compute_ivs
from wiener_chaos.core import chaos as general_chaos_module
general_chaos_module = importlib.reload(general_chaos_module)
from wiener_chaos.pricing import heston as heston_module
heston_module = importlib.reload(heston_module)
from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
from wiener_chaos.experiments.calibration import evaluate_surface, save_artifacts
ModelStock = pricing_module.ModelStock
print(f"Pricing module loaded: {pricing_module.__file__}")

DATA_PATH = ROOT / "Rough Heston" / "rough_heston_data.pkl"
with DATA_PATH.open("rb") as handle:
    rough_data = pickle.load(handle)

def surface_from_split(split):
    maturities = list(rough_data[f"maturities_{split}"])
    return {
        "S0": rough_data["S0"], "maturities": maturities,
        "strikes": rough_data[f"strikes_{split}"],
        "prices": rough_data[f"prices_{split}"],
        "iv": rough_data[f"iv_{split}"],
        "r_s": [rough_data["r"]] * len(maturities),
        "q_s": [rough_data["q"]] * len(maturities),
    }

data = surface_from_split("calibr")
test_data = surface_from_split("test")

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})


## 2. Calibration settings

The model uses `d=2`, `P=3`, `M=10`, a Legendre basis, and 32,768-path chunks.
Optimization and independent evaluation at the calibrated maturities both use
`[2_000_000, 1_000_000] + [200_000] * 2 + [100_000] * 6` paths
(**4 million** total). Evaluation counts are copied from the optimization counts;
the samples use separate seeds.

The Heston projection uses 5,000 × 128 paths and 201 time-grid points. Chaos
calibration uses `lr=1e-4`, at most 600 steps, patience 60, resampling every 50
steps, degree-2 moment control variates, and zero regularization weights.
Validation monitoring is disabled. Held-out pricing uses 20 million paths at the
shortest maturity, 2 million at each of the next two, and 500,000 at each remaining
maturity, processed in batches of at most 4 million paths.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.



In [ ]:
CUDA_DEVICE_INDEX = 0
if torch.cuda.is_available() and not 0 <= CUDA_DEVICE_INDEX < torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(1)

N_MATURITIES = len(data["maturities"])
CONFIG = {
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Algorithm for evaluating conditional chaos moments.
    "backend": "tensor",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 3,
    # Legendre basis: ten time functions per Brownian driver.
    "basis": "legendre",
    "M": 10,
    "base_jitter": 1e-12,
    # Paths evaluated per chunk by the general-basis pricing kernel.
    "pricing_chunk_size": 32_768,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        # General-basis projection currently supports the plain estimator.
        "projection_control": "none",
        # Antithetic projection is currently supported only for piecewise bases.
        "projection_antithetic": False,
        # Independent projection estimates averaged before normalization.
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 5_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 128,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 201,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 500,
            # Heston optimizer tolerance for objective improvement.
            "ftol": 1e-12,
            # Heston optimizer tolerance for the gradient.
            "gtol": 1e-9,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["MC"] * 10,
        # MC paths per calibration maturity; None for quadrature.
        # More paths at the shortest maturity to resolve deep OTM payoffs.
        "N": [2_000_000] + [1_000_000] + [200_000] * 2 + [100_000] * 6,
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 10,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 10,
    },
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 600,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 60,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 2e-8,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 2e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # MC paths reserved to estimate control-variate weights.
        #"n_samples_cv": 10_000,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": 25,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
# Use the same per-maturity path counts with independent evaluation draws.
CONFIG["comparison_N_per_mat"] = list(CONFIG["pricing_calibr"]["N"])
OUTPUT_DIR = STUDY_DIR / "Statistics calibration" / "single"
FIXED_VALIDATION_SEED = 4_000_042
FIXED_EVALUATION_SEED = 5_000_042

# Held-out evaluation; batch_size limits paths held in memory.
N_TEST_MATURITIES = len(test_data["maturities"])
CONFIG["pricing_test"] = {
    "method": ["MC"] * N_TEST_MATURITIES,
    "N": [20_000_000] + [2_000_000] * 2 + [500_000] * 4,
    "batch_size": 4_000_000,
    "Ns": [1] * N_TEST_MATURITIES,
    "nodes": [None] * N_TEST_MATURITIES,
}
FIXED_TEST_SEED = 6_000_042

for label, surface, allocation in (
    ("calibration", data, CONFIG["pricing_calibr"]),
    ("held-out", test_data, CONFIG["pricing_test"]),
):
    n = len(surface["maturities"])
    for key in ("strikes", "prices", "iv", "r_s", "q_s"):
        if len(surface[key]) != n:
            raise ValueError(f"{label}: {key} must contain one entry per maturity.")
    for rows in zip(surface["strikes"], surface["prices"], surface["iv"]):
        if len({len(row) for row in rows}) != 1:
            raise ValueError(f"{label}: strikes, prices, and IVs must be aligned.")
    for key in ("method", "N", "Ns", "nodes"):
        if len(allocation[key]) != n:
            raise ValueError(f"{label}: pricing {key} must have one entry per maturity.")
if len(CONFIG["comparison_N_per_mat"]) != N_MATURITIES:
    raise ValueError("comparison_N_per_mat must contain one entry per maturity.")
if set(data["maturities"]) & set(test_data["maturities"]):
    raise ValueError("Held-out maturities must differ from calibration maturities.")

print(f"Device: {DEVICE} | Basis: {CONFIG['basis']} | Order: {CONFIG['P']} | "
      f"Functions per driver: {CONFIG['M']} | Initialization: {CONFIG['initialization']['type']}")
print(f"Training paths: {sum(CONFIG['pricing_calibr']['N']):,} | "
      f"Independent evaluation paths: {sum(CONFIG['comparison_N_per_mat']):,} | "
      f"Pricing chunk size: {CONFIG['pricing_chunk_size']:,}")
print(f"Results: {OUTPUT_DIR}")
display(pd.DataFrame({
    "maturity": data["maturities"],
    "quotes": [len(row) for row in data["strikes"]],
    "rate": data["r_s"],
    "dividend_yield": data["q_s"],
    "pricing": CONFIG["pricing_calibr"]["method"],
    "training_N": CONFIG["pricing_calibr"]["N"],
    "validation_evaluation_N": CONFIG["comparison_N_per_mat"],
}))

display(pd.DataFrame({
    "held_out_maturity": test_data["maturities"],
    "quotes": [len(row) for row in test_data["strikes"]],
    "evaluation_N": CONFIG["pricing_test"]["N"],
}))


## 3. Calibration routines

These cells define the workflow helpers. They do not start calibration.

### Reproducibility, saving, and error metrics

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value):
    """Detach nested tensors and use JSON null for nonfinite diagnostics."""
    if torch.is_tensor(value):
        return json_safe(value.detach().cpu().tolist())
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(payload), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def surface_metrics(actual, expected):
    """Do not silently discard invalid IVs or mismatched surface entries."""
    if len(actual) != len(expected):
        raise ValueError("Different numbers of maturity rows.")
    errors = []
    invalid = 0
    for row, reference in zip(actual, expected):
        if len(row) != len(reference):
            raise ValueError("Different numbers of strikes.")
        for value, target in zip(row, reference):
            if value is None or not np.isfinite(value) or not np.isfinite(target):
                invalid += 1
            else:
                errors.append(abs(float(value) - float(target)))
    return {
        "mae": float(np.mean(errors)) if errors and not invalid else None,
        "invalid_quotes": invalid,
    }

### General-basis model and independent evaluation

Build one model over the full time horizon and register both sets of maturities.
Only calibration quotes enter initialization and optimization. The comparison
allocation applies at calibration maturities; test maturities use their own
allocation. Seeded evaluations preserve the training random stream.


In [ ]:
def initialize_parameters_timed(model, **kwargs):
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    started = time.perf_counter()
    initial = model.initialize_params(**kwargs)
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    return initial, time.perf_counter() - started


def build_model(data, test_data, config):
    model = ModelStock(data["S0"], torch.device(config["device"]), torch.float32)
    model.initialize_chaos(
        config["basis"], max(data["maturities"] + test_data["maturities"]), config["d"], config["P"], config["M"],
        base_jitter=config["base_jitter"], backend=config["backend"],
    )
    model.chaos.chunk_size = int(config["pricing_chunk_size"])
    if model.chaos.chunk_size < 1:
        raise ValueError("pricing_chunk_size must be positive.")
    model.set_up_for_Ts(data["maturities"], data["r_s"], data["q_s"])
    model.set_up_for_Ts(test_data["maturities"], test_data["r_s"], test_data["q_s"])
    # Seeded validation and final evaluation use independent comparison samples.
    comparison_N = {round(float(T), 6): count for T, count in zip(
        data["maturities"], config["comparison_N_per_mat"],
    )}
    original_seeded_sample = model._simulate_X_block_with_seed

    def fixed_comparison_sample(maturities_block, method_block, N_block, Ns_block, seed=None):
        if seed is not None:
            # Keep requested allocations for held-out maturities.
            N_block = [comparison_N.get(round(float(T), 6), count)
                       for T, count in zip(maturities_block, N_block)]
        return original_seeded_sample(
            maturities_block, method_block, N_block, Ns_block, seed,
        )

    model._simulate_X_block_with_seed = fixed_comparison_sample
    return model


def fitted_heston_surface(params, data):
    prices = [
        heston_module.heston_price_vec(params, r, q, T, data["S0"], strikes).tolist()
        for T, strikes, r, q in zip(data["maturities"], data["strikes"], data["r_s"], data["q_s"])
    ]
    ivs = compute_ivs(prices, data["maturities"], data["strikes"],
                      data["S0"], data["r_s"], data["q_s"])
    return prices, ivs


## 4. Run one calibration

Fit Heston to the rough-Heston calibration quotes and project its terminal factor,
or choose random initialization. Save the initial coefficients before training
modifies them. Compare the fitted Heston and projected start to the final model
on calibration quotes, then price the held-out maturities without further fitting.

Initialization, training, calibration-surface evaluation, and held-out evaluation
use separate reproducible seeds. An iteration-cap message flags runs that should
be inspected before concluding convergence.


In [ ]:
progress = display(Markdown("Initializing the rough-Heston Legendre model…"), display_id=True)
seed = CONFIG["base_seed"]
calibration_seed = seed + 3_000_000
seed_everything(seed)
model = build_model(data, test_data, CONFIG)
pricing = CONFIG["pricing_calibr"]
init_config = CONFIG["initialization"]
fit = None
prices_heston = iv_heston = None
if init_config["type"] == "calibrate_heston":
    initial, init_seconds = initialize_parameters_timed(
        model, type="calibrate_heston", maturities=data["maturities"],
        strikes=data["strikes"], prices_market=data["prices"],
        r=data["r_s"], q=data["q_s"], seed=seed,
        **{key: value for key, value in init_config.items() if key != "type"},
    )
    fit = model.heston_fit_result
    if fit is not None:
        print(f"Heston optimizer success: {fit.success}; iterations: {fit.nit}; "
              f"objective: {fit.fun:.6e}; {fit.message}")
    else:
        print("Using heuristic Heston parameters; optimizer skipped.")
    print(f"Heston parameters (kappa, theta, sigma, rho, v0): {model.heston_params}")
    prices_heston, iv_heston = fitted_heston_surface(model.heston_params, data)
elif init_config["type"] == "random":
    initial, init_seconds = initialize_parameters_timed(
        model, type="random", scale=CONFIG["init_scale"],
    )
else:
    raise ValueError(f"Unknown initialization type: {init_config['type']}")
initial_coefficients = initial.detach().cpu().clone()
progress.update(Markdown("Evaluating the starting chaos model…"))
prices_initial, iv_initial = evaluate_surface(
    model, initial, data, pricing, CONFIG["calibration"], FIXED_EVALUATION_SEED,
)

progress.update(Markdown("Calibrating the Legendre coefficients…"))
seed_everything(calibration_seed)
d_a, prices_model, loss_ins, loss_oos, fit_seconds = model.calibrate(
    data["iv"], data["prices"], [data["maturities"]], data["strikes"],
    pricing["method"], pricing["N"], pricing["Ns"], pricing["nodes"],
    d_a=initial, validation_seed=FIXED_VALIDATION_SEED,
    evaluation_seed=FIXED_EVALUATION_SEED, **CONFIG["calibration"],
)
if not torch.isfinite(d_a).all():
    raise FloatingPointError("Nonfinite calibrated coefficients.")
iv_model = compute_ivs(prices_model, data["maturities"], data["strikes"],
                       data["S0"], data["r_s"], data["q_s"])
progress.update(Markdown("Pricing the held-out maturities…"))
if model.device.type == "cuda":
    torch.cuda.synchronize(model.device)
test_started = time.perf_counter()
prices_model_test, iv_model_test = evaluate_surface(
    model, d_a, test_data, CONFIG["pricing_test"], CONFIG["calibration"], FIXED_TEST_SEED,
)
if model.device.type == "cuda":
    torch.cuda.synchronize(model.device)
test_seconds = time.perf_counter() - test_started

monitor_name = ("best_training_price_mse" if CONFIG["calibration"]["check_validation_set_MC"] is None
                else "best_validation_price_mse")
result = json_safe({
    "config": CONFIG, "seed": seed, "projection_seed": seed,
    "calibration_seed": calibration_seed,
    "validation_seed": (FIXED_VALIDATION_SEED
                        if CONFIG["calibration"]["check_validation_set_MC"] is not None else None),
    "evaluation_seed": FIXED_EVALUATION_SEED, "test_seed": FIXED_TEST_SEED,
    "training_N_per_mat": pricing["N"],
    "comparison_N_per_mat": CONFIG["comparison_N_per_mat"],
    "d_a": d_a, "initial_coefficients": initial_coefficients,
    "heston_init_params": getattr(model, "heston_params", None),
    "heston_initial_guess": getattr(model, "heston_initial_guess", None),
    "heston_fit": ({"success": bool(fit.success), "iterations": int(fit.nit),
                    "objective": float(fit.fun), "message": str(fit.message)} if fit is not None else None),
    "maturities": data["maturities"], "r_s": data["r_s"], "q_s": data["q_s"],
    "loss_ins": loss_ins, "loss_oos": loss_oos,
    "calibration_iterations": model.last_calibration_iterations,
    monitor_name: model.last_calibration_monitor_loss,
    "prices_heston": prices_heston, "iv_heston": iv_heston,
    "prices_initial": prices_initial, "iv_initial": iv_initial,
    "prices_model": prices_model, "iv_model": iv_model,
    "maturities_test": test_data["maturities"],
    "r_s_test": test_data["r_s"], "q_s_test": test_data["q_s"],
    "prices_model_test": prices_model_test, "iv_model_test": iv_model_test,
    "iv_test_metrics": surface_metrics(iv_model_test, test_data["iv"]),
    "price_calibration_metrics": surface_metrics(prices_model, data["prices"]),
    "price_test_metrics": surface_metrics(prices_model_test, test_data["prices"]),
    "iv_heston_metrics": surface_metrics(iv_heston, data["iv"]) if iv_heston is not None else None,
    "iv_initial_metrics": surface_metrics(iv_initial, data["iv"]),
    "iv_calibration_metrics": surface_metrics(iv_model, data["iv"]),
    "elapsed_seconds_init": init_seconds,
    "elapsed_seconds_chaos_calibration": fit_seconds,
    # Same fitting-time definition as SPX; held-out pricing is reported separately.
    "elapsed_seconds_total": init_seconds + fit_seconds,
    "elapsed_seconds_test_evaluation": test_seconds,
    "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "source_sha256": {name: source_fingerprint(ROOT / name)
                      for name in ("wiener_chaos/core/model.py", "wiener_chaos/pricing/heston.py", "wiener_chaos/core/chaos.py",
                                   "wiener_chaos/core/basis.py", "wiener_chaos/core/basis_integrals.py", "wiener_chaos/core/utils.py", "wiener_chaos/pricing/options.py",
                                   "wiener_chaos/experiments/calibration.py", "wiener_chaos/experiments/provenance.py")},
    "notebook_source_sha256": hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest(),
    "runtime": {"torch": torch.__version__, "numpy": np.__version__, "device": CONFIG["device"]},
})
save_json(OUTPUT_DIR / "result.json", result)
mae = result["iv_calibration_metrics"]["mae"]
mae_text = f"{mae * 1e4:.2f} bp" if mae is not None else "unavailable (invalid IVs)"
progress.update(Markdown(
    f"Initialization: {init_seconds:.2f}s | Calibration: {fit_seconds:.2f}s | "
    f"Iterations: {model.last_calibration_iterations} | Calibration IV MAE: {mae_text}"
))
test_mae = result["iv_test_metrics"]["mae"]
print("Held-out IV MAE: " + (f"{test_mae * 1e4:.2f} bp" if test_mae is not None
                              else "unavailable (invalid IVs)"))
print(f"Held-out pricing: {test_seconds:.2f}s")
if model.last_calibration_iterations >= CONFIG["calibration"]["n_iter"]:
    print("The iteration cap was reached; inspect convergence before interpreting this fit.")


## 5. Accuracy and runtime

Compare fitted Heston, the starting chaos model, and calibrated Legendre at
calibration maturities. Report held-out Legendre accuracy separately.
IV MAE is in basis points (`1 bp = 0.0001`); price MAE is in spot-price units.
An invalid IV invalidates that surface's IV MAE rather than being silently removed.

Starting and final chaos prices at calibration maturities share independent Monte
Carlo draws. Initialization includes the Heston fit and projection; calibration
includes training and final calibration-surface pricing inside `model.calibrate()`.
Their sum uses the SPX timing definition and excludes setup, baseline evaluations,
held-out pricing, tables, and plots. Held-out pricing time is reported separately.


In [ ]:
heston_label = "Fitted Heston" if fit is not None else "Heuristic Heston"
comparison_surfaces = {}
comparison_prices = {}
if iv_heston is not None:
    comparison_surfaces[heston_label] = iv_heston
    comparison_prices[heston_label] = prices_heston
start_label = "Projected Heston start" if init_config["type"] == "calibrate_heston" else "Random start"
comparison_surfaces[start_label] = iv_initial
comparison_prices[start_label] = prices_initial
comparison_surfaces["Calibrated Legendre"] = iv_model
comparison_prices["Calibrated Legendre"] = prices_model

fit_rows, summary_rows = [], []
surfaces_by_split = {
    "calibration": (data, comparison_surfaces, comparison_prices),
    "held-out": (test_data, {"Calibrated Legendre": iv_model_test},
                 {"Calibrated Legendre": prices_model_test}),
}
for split, (surface, iv_surfaces, price_surfaces) in surfaces_by_split.items():
    for label, fitted in iv_surfaces.items():
        iv_metrics = surface_metrics(fitted, surface["iv"])
        price_metrics = surface_metrics(price_surfaces[label], surface["prices"])
        summary_rows.append({
            "split": split, "model": label,
            "quotes": sum(map(len, surface["strikes"])),
            "iv_mae_bp": None if iv_metrics["mae"] is None else 1e4 * iv_metrics["mae"],
            "invalid_iv_quotes": iv_metrics["invalid_quotes"],
            "price_mae": price_metrics["mae"],
        })
        for T, row, target in zip(surface["maturities"], fitted, surface["iv"]):
            metrics = surface_metrics([row], [target])
            fit_rows.append({
                "split": split, "model": label, "maturity": T, "quotes": len(row),
                "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
                "invalid_iv_quotes": metrics["invalid_quotes"],
            })
fit_summary = pd.DataFrame(summary_rows).set_index(["split", "model"])
fit_by_maturity = pd.DataFrame(fit_rows).set_index(["split", "model", "maturity"])
display(fit_summary.round(4))
display(fit_by_maturity.round(4))
print(f"Initialization: {init_seconds:.2f}s | Calibration: {fit_seconds:.2f}s | "
      f"Combined fitting: {init_seconds + fit_seconds:.2f}s | "
      f"Iterations: {model.last_calibration_iterations} | Held-out pricing: {test_seconds:.2f}s")


In [ ]:
def plot_rough_smiles(surface, surfaces, split):
    columns = 3
    rows = (len(surface["maturities"]) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.4 * rows), squeeze=False)
    styles = {
        "Fitted Heston": {"linestyle": "--", "color": "tab:orange"},
        "Heuristic Heston": {"linestyle": "--", "color": "tab:orange"},
        "Projected Heston start": {"linestyle": ":", "color": "tab:green"},
        "Random start": {"linestyle": ":", "color": "tab:green"},
        "Calibrated Legendre": {"color": "tab:red"},
    }
    for index, (ax, T, strikes, target, r, q) in enumerate(zip(
        axes.flat, surface["maturities"], surface["strikes"], surface["iv"],
        surface["r_s"], surface["q_s"],
    )):
        forward = surface["S0"] * np.exp((r - q) * T)
        x = np.log(np.asarray(strikes) / forward)
        ax.plot(x, 100 * np.asarray(target), ".", color="tab:blue", label="Rough-Heston target")
        for label, fitted in surfaces.items():
            ax.plot(x, 100 * np.asarray(fitted[index], dtype=float),
                    label=label, **styles[label])
        ax.set(title=f"T = {T:g}", xlabel="log(K / forward)", ylabel="Implied volatility (%)")
    for ax in list(axes.flat)[len(surface["maturities"]):]:
        ax.set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(.5, 1.01))
    fig.suptitle(f"Rough Heston — Legendre ({split})", y=1.04)
    fig.tight_layout()
    plt.show()
    return fig

figures = {
    "calibration": plot_rough_smiles(data, comparison_surfaces, "calibration"),
    "test": plot_rough_smiles(
        test_data, {"Calibrated Legendre": iv_model_test}, "held-out maturities",
    ),
}


## 6. Saved results

`Statistics calibration/single/result.json` records the configuration, fitted
coefficients, prices, implied volatilities, seeds, timings, and provenance.
`coefficients/` contains `initial_coefficients.pt` and `calibrated_coefficients.pt`;
`tables/` contains `fit_summary.csv` and `fit_by_maturity.csv`. Comparison figures
are saved in `figures/` as PNG and PDF files.


In [ ]:
save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": result["initial_coefficients"],
                  "calibrated_coefficients": result["d_a"]},
    tables={"fit_summary": fit_summary, "fit_by_maturity": fit_by_maturity},
    figures=figures,
)
print(f"Single-calibration results saved in: {OUTPUT_DIR}")
